In [1]:
import fastf1 as f1
import pandas as pd

In [2]:
f1.Cache.enable_cache("fastf1_cache")

In [3]:
session = f1.get_session(2024, 'Melbourne' , 'R')

In [4]:
session.load(telemetry = False, weather = True, messages = False)

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 19 drivers: ['55', '16', '4', '81', '11', '18', '22', '14', '27', '20', '23', '3', '10', '77', '24', '31', '63', '44', '1']


In [5]:
laps = session.laps[['LapTime', 'Position', 'Driver', 'Team','LapNumber' , 'Stint', 'Compound', 'FreshTyre', 'TrackStatus', 'IsAccurate']]

In [6]:
df = pd.DataFrame(laps)

In [7]:
df['LapTime'] = df['LapTime'].dt.total_seconds()

In [8]:
df.shape

(998, 10)

In [9]:
df['PitIn'] = session.laps['PitInTime'].notna()

In [10]:
df = df.sort_values(['Driver', 'LapNumber'])

In [11]:
temp = pd.DataFrame(session.laps)

In [12]:
temp = temp.sort_values(['Driver' , 'LapNumber'])

In [13]:
temp['NextPitOutTime'] = temp.groupby('Driver')['PitOutTime'].shift(-1)

In [14]:
temp['PitDuration'] = (temp['NextPitOutTime'] - temp['PitInTime']).dt.total_seconds()

In [15]:
df = df.merge(temp[['Driver', 'LapNumber' , 'PitDuration']] , on = ['Driver', 'LapNumber'], how='left')

In [16]:
df['PitDuration'] = df['PitDuration'].fillna(0.0)

In [17]:
df

,LapTime,Position,Driver,Team,LapNumber,Stint,Compound,FreshTyre,TrackStatus,IsAccurate,PitIn,PitDuration
0,95.752,14.0,ALB,Williams,1.0,1.0,MEDIUM,True,1,False,False,0.0
1,84.536,14.0,ALB,Williams,2.0,1.0,MEDIUM,True,1,True,False,0.0
2,83.606,14.0,ALB,Williams,3.0,1.0,MEDIUM,True,1,True,False,0.0
3,83.438,13.0,ALB,Williams,4.0,1.0,MEDIUM,True,1,True,False,0.0
4,83.916,13.0,ALB,Williams,5.0,1.0,MEDIUM,True,1,True,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
993,82.184,16.0,ZHO,Kick Sauber,53.0,3.0,HARD,True,1,True,False,0.0
994,82.067,16.0,ZHO,Kick Sauber,54.0,3.0,HARD,True,1,True,False,0.0
995,81.798,16.0,ZHO,Kick Sauber,55.0,3.0,HARD,True,1,True,False,0.0
996,88.058,16.0,ZHO,Kick Sauber,56.0,3.0,HARD,True,126,False,False,0.0


In [18]:
del temp

In [19]:
start = session.laps['LapStartTime'].min()

In [20]:
start

Timedelta('0 days 00:58:29.232000')

In [21]:
weather = session.weather_data[session.weather_data['Time'] <= start].iloc[[1]]

In [22]:
w = pd.DataFrame(weather)

In [23]:
w

,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
1,0 days 00:01:08.796000,19.0,48.0,1021.6,False,37.0,0,0.5


In [24]:
w = w.drop(columns = 'Time')

In [25]:
w

,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
1,19.0,48.0,1021.6,False,37.0,0,0.5


In [26]:
df['Rain'] = w['Rainfall'].iloc[0]

In [27]:
df

,LapTime,Position,Driver,Team,LapNumber,Stint,Compound,FreshTyre,TrackStatus,IsAccurate,PitIn,PitDuration,Rain
0,95.752,14.0,ALB,Williams,1.0,1.0,MEDIUM,True,1,False,False,0.0,False
1,84.536,14.0,ALB,Williams,2.0,1.0,MEDIUM,True,1,True,False,0.0,False
2,83.606,14.0,ALB,Williams,3.0,1.0,MEDIUM,True,1,True,False,0.0,False
3,83.438,13.0,ALB,Williams,4.0,1.0,MEDIUM,True,1,True,False,0.0,False
4,83.916,13.0,ALB,Williams,5.0,1.0,MEDIUM,True,1,True,False,0.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
993,82.184,16.0,ZHO,Kick Sauber,53.0,3.0,HARD,True,1,True,False,0.0,False
994,82.067,16.0,ZHO,Kick Sauber,54.0,3.0,HARD,True,1,True,False,0.0,False
995,81.798,16.0,ZHO,Kick Sauber,55.0,3.0,HARD,True,1,True,False,0.0,False
996,88.058,16.0,ZHO,Kick Sauber,56.0,3.0,HARD,True,126,False,False,0.0,False


In [28]:
df.shape

(998, 13)

In [29]:
del w

In [30]:
df['OutLap'] = df.groupby('Driver')['PitIn'].shift(1,fill_value = False) 

In [31]:
df = df[df['TrackStatus'] == '1']

In [32]:
df.shape

(937, 14)

In [33]:
df

,LapTime,Position,Driver,Team,LapNumber,Stint,Compound,FreshTyre,TrackStatus,IsAccurate,PitIn,PitDuration,Rain,OutLap
0,95.752,14.0,ALB,Williams,1.0,1.0,MEDIUM,True,1,False,False,0.0,False,False
1,84.536,14.0,ALB,Williams,2.0,1.0,MEDIUM,True,1,True,False,0.0,False,False
2,83.606,14.0,ALB,Williams,3.0,1.0,MEDIUM,True,1,True,False,0.0,False,False
3,83.438,13.0,ALB,Williams,4.0,1.0,MEDIUM,True,1,True,False,0.0,False,False
4,83.916,13.0,ALB,Williams,5.0,1.0,MEDIUM,True,1,True,False,0.0,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
991,81.830,16.0,ZHO,Kick Sauber,51.0,3.0,HARD,True,1,True,False,0.0,False,False
992,82.060,16.0,ZHO,Kick Sauber,52.0,3.0,HARD,True,1,True,False,0.0,False,False
993,82.184,16.0,ZHO,Kick Sauber,53.0,3.0,HARD,True,1,True,False,0.0,False,False
994,82.067,16.0,ZHO,Kick Sauber,54.0,3.0,HARD,True,1,True,False,0.0,False,False


In [34]:
df = df.drop(columns='IsAccurate')

In [35]:
df

,LapTime,Position,Driver,Team,LapNumber,Stint,Compound,FreshTyre,TrackStatus,PitIn,PitDuration,Rain,OutLap
0,95.752,14.0,ALB,Williams,1.0,1.0,MEDIUM,True,1,False,0.0,False,False
1,84.536,14.0,ALB,Williams,2.0,1.0,MEDIUM,True,1,False,0.0,False,False
2,83.606,14.0,ALB,Williams,3.0,1.0,MEDIUM,True,1,False,0.0,False,False
3,83.438,13.0,ALB,Williams,4.0,1.0,MEDIUM,True,1,False,0.0,False,False
4,83.916,13.0,ALB,Williams,5.0,1.0,MEDIUM,True,1,False,0.0,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
991,81.830,16.0,ZHO,Kick Sauber,51.0,3.0,HARD,True,1,False,0.0,False,False
992,82.060,16.0,ZHO,Kick Sauber,52.0,3.0,HARD,True,1,False,0.0,False,False
993,82.184,16.0,ZHO,Kick Sauber,53.0,3.0,HARD,True,1,False,0.0,False,False
994,82.067,16.0,ZHO,Kick Sauber,54.0,3.0,HARD,True,1,False,0.0,False,False


In [ ]:
df['Compound'] = df['Compound'].replace(['ULTRASOFT','HYPERSOFT','SUPERSOFT'],'SOFT')